# Stretch: making the pipeline sturdier

For when the live exercise is done and there is time left. These get
harder as you go, so nobody is expected to finish them all.

Each one adds one thing a real pipeline needs, and each one is a function
you write and check here, then could add to a script. The rules from today
still hold: functions return, only the outer layer prints, every function
gets a docstring. Uncomment the `assert` lines once your function exists.

Two of them need something new, and the notebook tells you what it is when
you get there.

## Start here

This imports today's genre pipeline, so you can call its functions
(`load_messy`, `clean_plays`, `summarise_by_genre`, `save_csv`, ...) as
`gp.load_messy` and so on.

In [ ]:
import os
import sqlite3
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-10/demos")
import genre_pipeline as gp

raw = gp.load_messy(gp.MESSY)
plays = gp.clean_plays(raw)
clean = pd.read_csv("data/clean/plays.csv")
print(len(raw), "raw rows,", len(plays), "cleaned")

## Stretch 1: swap the source

The same log lives in `data/music.db`, undamaged, split over two tables.
Write `load_from_db(path)`, which returns a DataFrame with **exactly the
columns and types of `clean`**, using one SQL query with a join (session 7)
and `pd.read_sql` (session 8).

The database stores `played_at` as `YYYY-MM-DD` text, just like the clean
file, so no conversion is needed.

In [ ]:
# TODO: def load_from_db(path):


# from_db = load_from_db("data/music.db")
# assert from_db.shape == (2183, 9)
# assert list(from_db.columns) == list(clean.columns)
# assert from_db.dtypes.equals(clean.dtypes)

Now feed it straight into `gp.summarise_by_genre`, with no cleaning step.
Compare Electronic's minutes with today's 2,753.2. Which is right, and why
do they differ?

In [ ]:
# TODO

*Your answer here.*

## Stretch 2: a cleaning report

`cleaning_report(raw, plays)` returns a dictionary saying what cleaning did,
so a run can tell you instead of you having to dig:

```
{"rows in": 2223, "duplicates removed": 40, "rows out": 2183,
 "minutes missing": 94, "devices unknown": 69}
```

It must work these numbers out from the two DataFrames, not have them
typed in.

In [ ]:
# TODO: def cleaning_report(raw, plays):


# report = cleaning_report(raw, plays)
# assert report["duplicates removed"] == 40
# assert report["minutes missing"] == 94
# assert report["devices unknown"] == 69

Write the line of `main()` that would print it, one item per line.

In [ ]:
# TODO

## Stretch 3: fail early, with a clear message

Point the pipeline at the wrong file, say `data/clean/artists.csv`, and it
fails inside `clean_plays` with an error that sounds like a bug in the
code. Uncomment and run it:

In [ ]:
# gp.clean_plays(gp.load_messy("data/clean/artists.csv"))

Better to check at load time and say what is wrong in plain words.

**New tool:** `raise` stops the program with an error of your choosing:

```python
raise ValueError("a message that says what is wrong")
```

Write `check_columns(df, required)`. It returns nothing when every column in
`required` is in `df`, and raises a `ValueError` naming the missing ones
when not.

In [ ]:
REQUIRED = ["play_id", "played_at", "artist_name", "genre", "country",
            "device", "skipped", "minutes played "]

# TODO: def check_columns(df, required):


# check_columns(raw, REQUIRED)                                  # silent: all there
# check_columns(gp.load_messy("data/clean/artists.csv"), REQUIRED)  # a clear error

**Think about it:** why is a crash with a clear message *better* than a
pipeline that carries on? Who reads the message?

*Your answer here.*

## Stretch 4: one year, from the command line

Next month you will want this for 2025 only. A script can read words typed
after its name: `python pipeline.py 2025` gives it the list
`sys.argv == ["pipeline.py", "2025"]`, always text.

1. `parse_year(argv)` returns the year as a number, or `None` if no year was
   given.
2. `plays_in_year(plays, year)` returns only that year's plays, or all of
   them when `year` is `None`.

In [ ]:
# TODO: def parse_year(argv):


# TODO: def plays_in_year(plays, year):


# assert parse_year(["pipeline.py"]) is None
# assert parse_year(["pipeline.py", "2025"]) == 2025
# assert len(plays_in_year(plays, None)) == 2183
# assert len(plays_in_year(plays, 2025)) == 1061

In a script, `main()` would start with `year = parse_year(sys.argv)`. Why
is `parse_year` handed the list, instead of reading `sys.argv` itself?

*Your answer here.*

## Stretch 5: a cleaning decision as a parameter

`clean_plays` keeps rows with missing minutes. Session 9 showed that every
such decision changes the answer, so make it a choice. Write
`fill_minutes(plays, how="keep")` which returns a copy of the cleaned plays
where the missing minutes are:

* `"keep"`: left missing (what we do today)
* `"drop"`: the rows removed
* `"median"`: filled with the median of all known minutes

Then run `gp.summarise_by_genre` on all three, and compare Electronic's plays
and minutes with the true answer from stretch 1.

In [ ]:
# TODO: def fill_minutes(plays, how="keep"):


# assert len(fill_minutes(plays, "drop")) == 2089
# assert fill_minutes(plays, "median")["minutes_played"].isna().sum() == 0

In [ ]:
# TODO: compare the three

**Think about it:** one option gets the overall total closest to the truth.
Is that the one you would choose? What does each option make the chart
claim?

*Your answer here.*

## Stretch 6: a log of every run

A pipeline runs many times. Keep a history: `log_run(path, plays, summary)`
**adds one line** to a CSV each time it is called, with the time, the
number of plays and the top genre. If the file does not exist yet, it
writes the header first.

Two tools:

* `datetime.now().isoformat(timespec="seconds")`, after
  `from datetime import datetime`, gives the time as text
* `open(path, "a")` **appends**, where `"w"` would replace the file

In [ ]:
# TODO: def log_run(path, plays, summary):


# log = Path("output/runs.csv")
# summary = gp.summarise_by_genre(plays)
# before = len(pd.read_csv(log)) if log.exists() else 0
# log_run(log, plays, summary)
# log_run(log, plays, summary)
# assert len(pd.read_csv(log)) == before + 2
# pd.read_csv(log).tail(3)

**Think about it:** why `"a"` and not `"w"`? And what would you want this
file to tell you in six months, when a number suddenly looks different?

*Your answer here.*